# 06 — Guardrail Engine

Wires `physics/guardrails.py` + `physics/calculator.py` to the multihead surrogate.

**What this notebook does:**
1. Loads the trained `MultiHeadMGN` from `multihead_final.pt`
2. For each of the 80 HDF5 cases, extracts dimensionless numbers from stored global features
3. Adds **Eu** (Euler number) computed from the *predicted* pressure field
4. Runs `GuardrailEngine.check()` on every case
5. Visualises confidence scores and violation frequency across the sweep
6. Demonstrates the full pipeline: params → physics → surrogate → guardrail → confidence

**Key modules (already exist):**
- `physics/calculator.py` — 11 dimensionless number formulas
- `physics/guardrails.py` — `GuardrailEngine`, `GuardrailBounds`, `CheckResult`

**Requires:** Colab A100 GPU (for surrogate inference)

## 0. Setup

In [ ]:
import subprocess, sys, os

subprocess.run(['pip', 'install', '-q', 'torch_geometric', 'h5py', 'scipy'], check=True)

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

REPO_URL = 'https://github.com/Ranaam21/cfd-ald-app.git'
REPO_DIR = '/content/cfd-ald-app'
if os.path.exists(os.path.join(REPO_DIR, '.git')):
    r = subprocess.run(['git', '-C', REPO_DIR, 'pull'], capture_output=True, text=True)
    print(r.stdout.strip() or 'Repo up to date.')
else:
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
    print('Cloned.')

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print('Setup done.')

## 1. Paths + imports

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import h5py
from pathlib import Path
from tqdm import tqdm
from scipy.spatial import cKDTree

from physics.calculator import compute_all, k_rxn_from_sticking
from physics.guardrails import GuardrailEngine, GuardrailBounds

assert torch.cuda.is_available(), 'Switch to A100 GPU runtime'
DEVICE = torch.device('cuda')
print(f'GPU: {torch.cuda.get_device_name(0)}')

DRIVE_BASE    = Path('/content/drive/MyDrive/cfd-ald-app')
HDF5_DIR      = DRIVE_BASE / 'showerhead_openfoam'
MULTIHEAD_PT  = DRIVE_BASE / 'checkpoints' / 'multihead' / 'multihead_final.pt'
OUT_DIR       = DRIVE_BASE / 'checkpoints' / 'guardrail'
OUT_DIR.mkdir(parents=True, exist_ok=True)

h5_files = sorted(HDF5_DIR.glob('case_*.h5'))
print(f'Found {len(h5_files)} HDF5 files')
assert MULTIHEAD_PT.exists(), f'Missing: {MULTIHEAD_PT}'

# Global feature layout (from postprocess.py)
# [0]Re [1]Pr [2]Sc [3]Ma [4]Pe_h [5]Pe_m [6]Da
# [7]D [8]pitch_over_D [9]H_plenum [10]t_face [11]standoff
# [12]flow_rate_slm [13]beta [14]v_th [15]D_m [16]n_holes [17]open_area_frac
GLOBAL_KEYS = ['Re','Pr','Sc','Ma','Pe_h','Pe_m','Da',
                'D','pitch_over_D','H_plenum','t_face','standoff',
                'flow_rate_slm','beta','v_th','D_m','n_holes','open_area_frac']

OUTPUT_COLS = ['Ux', 'Uy', 'Uz', 'p', 'T', 'TMA']
P_IDX = 3

print('Imports OK.')

## 2. Load multihead model

In [ ]:
from torch_geometric.nn import MessagePassing

def mlp(in_dim, out_dim, hidden=None, n_layers=2):
    hidden = hidden or out_dim
    dims = [in_dim] + [hidden] * (n_layers - 1) + [out_dim]
    mods = []
    for i in range(len(dims) - 1):
        mods.append(nn.Linear(dims[i], dims[i+1]))
        if i < len(dims) - 2:
            mods.append(nn.SiLU())
    return nn.Sequential(*mods)

class MGNProcessor(MessagePassing):
    def __init__(self, hidden):
        super().__init__(aggr='sum')
        self.edge_mlp  = mlp(3*hidden, hidden)
        self.node_mlp  = mlp(2*hidden, hidden)
        self.edge_norm = nn.LayerNorm(hidden)
        self.node_norm = nn.LayerNorm(hidden)
    def forward(self, x, edge_index, edge_attr):
        src, dst = edge_index
        N = x.size(0)
        new_e = self.edge_norm(self.edge_mlp(torch.cat([x[src], x[dst], edge_attr], -1)) + edge_attr)
        agg   = self.propagate(edge_index, x=x, edge_attr=new_e, size=(N, N))
        new_x = self.node_norm(self.node_mlp(torch.cat([x, agg], -1)) + x)
        return new_x, new_e
    def message(self, edge_attr): return edge_attr

class MultiHeadMGN(nn.Module):
    def __init__(self, node_dim, edge_dim, flow_out=4, heat_out=1, species_out=1, hidden=128, n_layers=8):
        super().__init__()
        self.node_enc   = mlp(node_dim, hidden)
        self.edge_enc   = mlp(edge_dim, hidden)
        self.processors = nn.ModuleList([MGNProcessor(hidden) for _ in range(n_layers)])
        self.flow_dec    = mlp(hidden, flow_out)
        self.heat_dec    = mlp(hidden, heat_out)
        self.species_dec = mlp(hidden, species_out)
    def forward(self, x, edge_index, edge_attr):
        x = self.node_enc(x)
        e = self.edge_enc(edge_attr)
        for proc in self.processors:
            x, e = proc(x, edge_index, e)
        return self.flow_dec(x), self.heat_dec(x), self.species_dec(x)

# Load checkpoint
ckpt     = torch.load(MULTIHEAD_PT, map_location='cpu')
cfg      = ckpt['cfg']
norm     = ckpt['norm']
node_mean = np.array(norm['node_mean'], dtype=np.float32)
node_std  = np.array(norm['node_std'],  dtype=np.float32)
out_mean  = np.array(norm['out_mean'],  dtype=np.float32)
out_std   = np.array(norm['out_std'],   dtype=np.float32)

model = MultiHeadMGN(
    node_dim    = cfg['node_input_dim'],
    edge_dim    = cfg['edge_input_dim'],
    flow_out    = cfg['flow_out_dim'],
    heat_out    = cfg['heat_out_dim'],
    species_out = cfg['species_out_dim'],
    hidden      = cfg['hidden_dim'],
    n_layers    = cfg['n_layers'],
).to(DEVICE)
model.load_state_dict(ckpt['model'])
model.eval()

n_params = sum(p.numel() for p in model.parameters())
print(f'MultiHeadMGN loaded: {n_params/1e6:.2f}M params')
print(f'Final val loss: {ckpt["final_val_loss"]:.4f}')

## 3. Inference helper

Given one HDF5 case, builds the k-NN graph, runs the surrogate, and returns denormalised pressure field.

In [ ]:
K = cfg['k_neighbors']

def predict_case(h5_path: Path) -> dict:
    """Load HDF5, run surrogate, return dict with coords, predicted fields, global_feats."""
    with h5py.File(h5_path, 'r') as f:
        coords       = f['coords'][:].astype(np.float32)
        node_feats   = f['inputs/node_features'][:].astype(np.float32)
        global_feats = f['inputs/global'][:].astype(np.float32)   # [18]
        out_true     = f['outputs/node_fields'][:].astype(np.float32)

    N = len(coords)
    global_broadcast = np.tile(global_feats, (N, 1))
    node_input = np.concatenate([node_feats, global_broadcast], axis=1)

    # k-NN graph
    tree = cKDTree(coords)
    _, idx = tree.query(coords, k=K+1)
    idx = idx[:, 1:]
    src = np.repeat(np.arange(N), K)
    dst = idx.flatten()
    diff = coords[dst] - coords[src]
    dist = np.linalg.norm(diff, axis=1, keepdims=True)
    med  = float(np.median(dist)) + 1e-8
    edge_feats = np.concatenate([diff/med, dist/med], axis=1).astype(np.float32)

    x  = torch.from_numpy((node_input - node_mean) / node_std).to(DEVICE)
    ei = torch.tensor(np.stack([src, dst]), dtype=torch.long).to(DEVICE)
    ea = torch.from_numpy(edge_feats).to(DEVICE)

    with torch.no_grad():
        fp, hp, sp = model(x, ei, ea)
        fp = fp.cpu().numpy()
        hp = hp.cpu().numpy().flatten()
        sp = sp.cpu().numpy().flatten()

    # Denormalize
    p_pred = fp[:, P_IDX] * out_std[P_IDX] + out_mean[P_IDX]   # pressure [Pa]
    T_pred = hp * out_std[4] + out_mean[4]

    del x, ei, ea
    torch.cuda.empty_cache()

    return {
        'name':         h5_path.stem,
        'coords':       coords,
        'global_feats': global_feats,
        'p_pred':       p_pred,
        'T_pred':       T_pred,
        'p_true':       out_true[:, P_IDX],
    }

# Smoke test on one case
sample = predict_case(h5_files[0])
print(f"Case: {sample['name']}")
print(f"  p_pred: {sample['p_pred'].min():.2f} – {sample['p_pred'].max():.2f} Pa")
print(f"  T_pred: {sample['T_pred'].min():.2f} – {sample['T_pred'].max():.2f} K")

## 4. Batch guardrail check — all 80 cases

For each case:
- Re, Pr, Sc, Ma, Pe_h, Pe_m, Da → directly from HDF5 global features (computed by postprocess.py)
- **Eu** → computed from *predicted* pressure drop: Δp = p_max − p_min

In [ ]:
# N2 properties at ~300 K (used for Eu denominator)
RHO_N2 = 1.145    # kg/m³ at 300 K, 1 atm

engine = GuardrailEngine()   # default bounds

records = []
print(f'Checking {len(h5_files)} cases...')

for h5_path in tqdm(h5_files):
    try:
        res = predict_case(h5_path)
    except Exception as e:
        records.append({'name': h5_path.stem, 'error': str(e)})
        continue

    gf = res['global_feats']
    gd = dict(zip(GLOBAL_KEYS, gf))

    # Dimensionless numbers already in global features
    dim_nums = {
        'Re':   float(gd['Re']),
        'Pr':   float(gd['Pr']),
        'Sc':   float(gd['Sc']),
        'Ma':   float(gd['Ma']),
        'Pe_h': float(gd['Pe_h']),
        'Pe_m': float(gd['Pe_m']),
        'Da':   float(gd['Da']),
    }

    # Eu from predicted pressure field
    delta_p = float(res['p_pred'].max() - res['p_pred'].min())
    # V = mean nozzle exit velocity from flow_rate_slm and geometry
    Q_slm = float(gd['flow_rate_slm'])
    Q_m3s = Q_slm * 1.667e-5          # slm → m³/s
    D     = float(gd['D'])             # nozzle diameter [m]
    n_h   = max(int(round(float(gd['n_holes']))), 1)
    A_noz = n_h * np.pi * (D/2)**2    # total nozzle area [m²]
    V_noz = Q_m3s / A_noz             # mean nozzle exit velocity [m/s]

    if delta_p > 0 and V_noz > 0:
        from physics.calculator import euler
        try:
            dim_nums['Eu'] = euler(delta_p, RHO_N2, V_noz)
        except ValueError:
            pass  # skip Eu if inputs are degenerate

    result = engine.check(dim_nums)

    records.append({
        'name':        res['name'],
        'dim_nums':    dim_nums,
        'confidence':  result.confidence,
        'passed':      result.passed,
        'n_violations': len(result.violations),
        'violations':  [v.reason_code for v in result.violations],
        'flags':       result.special_flags,
        'recs':        result.recommendations,
    })

ok  = [r for r in records if r.get('passed')]
bad = [r for r in records if not r.get('passed')]
print(f'\nResults: {len(ok)} passed  |  {len(bad)} with violations')

## 5. Visualise confidence + violation frequency

In [ ]:
import matplotlib.pyplot as plt
import json
from collections import Counter

valid = [r for r in records if 'confidence' in r]
conf  = [r['confidence'] for r in valid]
names = [r['name'] for r in valid]

# Violation frequency by reason code
all_viols = [v for r in valid for v in r['violations']]
viol_counts = Counter(all_viols)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1. Confidence histogram
axes[0].hist(conf, bins=20, color='steelblue', edgecolor='white')
axes[0].axvline(0.5, color='red', linestyle='--', label='CFD refinement threshold')
axes[0].set_xlabel('Confidence score')
axes[0].set_ylabel('Number of cases')
axes[0].set_title('Guardrail confidence — 80 cases')
axes[0].legend()

# 2. Confidence vs case index, colored by Q
Q_vals = [r['dim_nums'].get('Re', 0) for r in valid]  # use Re as proxy for Q ordering
sc = axes[1].scatter(range(len(conf)), conf, c=Q_vals, cmap='plasma', s=20)
axes[1].axhline(0.5, color='red', linestyle='--')
plt.colorbar(sc, ax=axes[1], label='Re')
axes[1].set_xlabel('Case index'); axes[1].set_ylabel('Confidence')
axes[1].set_title('Confidence per case (colour = Re)')

# 3. Violation frequency
if viol_counts:
    codes, cnts = zip(*sorted(viol_counts.items(), key=lambda x: -x[1]))
    axes[2].barh(codes, cnts, color='tomato')
    axes[2].set_xlabel('Number of cases')
    axes[2].set_title('Violation frequency by reason code')
else:
    axes[2].text(0.5, 0.5, 'No violations!', ha='center', va='center',
                 transform=axes[2].transAxes, fontsize=14)
    axes[2].set_title('Violation frequency')

plt.tight_layout()
plt.savefig(str(OUT_DIR / 'guardrail_summary.png'), dpi=120, bbox_inches='tight')
plt.show()

# Print summary stats
print(f'Confidence:  mean={np.mean(conf):.3f}  min={np.min(conf):.3f}  max={np.max(conf):.3f}')
print(f'Below 0.5 (need CFD refinement): {sum(c < 0.5 for c in conf)}/{len(conf)} cases')
if viol_counts:
    print('\nViolation counts:')
    for code, cnt in sorted(viol_counts.items(), key=lambda x: -x[1]):
        print(f'  {code:30s}: {cnt}')

## 6. Example: full pipeline on one case

Demonstrate the complete flow: case name → physics → surrogate → guardrail → report.

In [ ]:
# Pick an interesting case — highest Re (largest Q, smallest D)
highest_re = max(valid, key=lambda r: r['dim_nums'].get('Re', 0))
lowest_re  = min(valid, key=lambda r: r['dim_nums'].get('Re', 0))

for label, rec in [('Highest Re', highest_re), ('Lowest Re', lowest_re)]:
    print(f'\n{"="*60}')
    print(f'{label}: {rec["name"]}')
    print(f'{"="*60}')
    print('Dimensionless numbers:')
    for k, v in rec['dim_nums'].items():
        print(f'  {k:6s} = {v:.4g}')

    # Re-run engine to get full CheckResult with messages
    result = engine.check(rec['dim_nums'])
    print(f'\n{result.summary()}')

## 7. Bounds override demo

Shows how the user can tighten guardrails (e.g. restrict to laminar Re < 500)
and see more cases flagged.

In [ ]:
# Tighter bounds: restrict to laminar, ALD-safe Da
strict_bounds = GuardrailBounds(
    Re  = (1.0,   500.0),   # strict laminar only
    Ma  = (0.0,   0.1),     # very low Ma
    Da  = (0.01,  10.0),
    Eu  = (1.0,   30.0),
)
strict_engine = GuardrailEngine(strict_bounds)

strict_pass = 0
strict_fail = []
for r in valid:
    res = strict_engine.check(r['dim_nums'])
    if res.passed:
        strict_pass += 1
    else:
        strict_fail.append((r['name'], [v.reason_code for v in res.violations]))

print(f'Default bounds:  {len(ok)}/{len(valid)} pass')
print(f'Strict bounds:   {strict_pass}/{len(valid)} pass')
print(f'\nAdditionally flagged with strict bounds ({len(strict_fail)} cases):')
for name, codes in strict_fail[:10]:
    print(f'  {name[:40]:40s}  {codes}')

## 8. Save results

In [ ]:
import json

# Serialise to JSON (drop non-serialisable numpy types)
def _clean(obj):
    if isinstance(obj, (np.floating, np.integer)):
        return float(obj)
    if isinstance(obj, dict):
        return {k: _clean(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [_clean(i) for i in obj]
    return obj

out_path = OUT_DIR / 'guardrail_results.json'
with open(out_path, 'w') as f:
    json.dump(_clean(records), f, indent=2)
print(f'Saved {len(records)} records → {out_path}')

# Summary
print(f'\nGuardrail summary over {len(valid)} cases:')
print(f'  Passed (default bounds): {len(ok)}')
print(f'  Mean confidence:         {np.mean(conf):.3f}')
print(f'  Cases needing CFD ref.:  {sum(c < 0.5 for c in conf)}')
print('\nReady for 07_optimizer.ipynb')